In [40]:
import requests
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY = os.getenv("API_KEY")

cities = ["Munich", "Berlin", "Paris", "London", "Rome"]

data = []

for city in cities:
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        weather = response.json()
        
        data.append({
            "city": city,
            "temperature": weather["main"]["temp"],
            "humidity": weather["main"]["humidity"],
            "pressure": weather["main"]["pressure"],
            "description": weather["weather"][0]["description"],
            "timestamp": datetime.now()
        })
    else:
        print(f"Error fetching {city}: {response.status_code}")

df = pd.DataFrame(data)

df

,city,temperature,humidity,pressure,description,timestamp
0,Munich,30.77,43,1020,scattered clouds,2026-06-23 12:45:46.228581
1,Berlin,27.34,48,1022,broken clouds,2026-06-23 12:45:46.289888
2,Paris,34.49,47,1019,scattered clouds,2026-06-23 12:45:46.332271
3,London,28.04,68,1019,few clouds,2026-06-23 12:45:46.396685
4,Rome,21.29,91,1017,overcast clouds,2026-06-23 12:45:46.436298


In [41]:
df.to_csv("weather_raw.csv", index=False)

In [42]:
import pandas as pd

df = pd.read_csv("weather_raw.csv")
df.head()

,city,temperature,humidity,pressure,description,timestamp
0,Munich,30.77,43,1020,scattered clouds,2026-06-23 12:45:46.228581
1,Berlin,27.34,48,1022,broken clouds,2026-06-23 12:45:46.289888
2,Paris,34.49,47,1019,scattered clouds,2026-06-23 12:45:46.332271
3,London,28.04,68,1019,few clouds,2026-06-23 12:45:46.396685
4,Rome,21.29,91,1017,overcast clouds,2026-06-23 12:45:46.436298


In [43]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   city         5 non-null      object 
 1   temperature  5 non-null      float64
 2   humidity     5 non-null      int64  
 3   pressure     5 non-null      int64  
 4   description  5 non-null      object 
 5   timestamp    5 non-null      object 
dtypes: float64(1), int64(2), object(3)
memory usage: 372.0+ bytes


city           0
temperature    0
humidity       0
pressure       0
description    0
timestamp      0
dtype: int64

In [44]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'] = df['timestamp'].dt.floor('s')
df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   city         5 non-null      object        
 1   temperature  5 non-null      float64       
 2   humidity     5 non-null      int64         
 3   pressure     5 non-null      int64         
 4   description  5 non-null      object        
 5   timestamp    5 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 372.0+ bytes


In [45]:
df.columns = df.columns.str.lower()
df.columns

Index(['city', 'temperature', 'humidity', 'pressure', 'description',
       'timestamp'],
      dtype='object')

In [46]:
df['temp_category'] = df['temperature'].apply(
    lambda x: 'Cold' if x < 10 else 'Moderate' if x < 25 else 'Hot')
df

,city,temperature,humidity,pressure,description,timestamp,temp_category
0,Munich,30.77,43,1020,scattered clouds,2026-06-23 12:45:46,Hot
1,Berlin,27.34,48,1022,broken clouds,2026-06-23 12:45:46,Hot
2,Paris,34.49,47,1019,scattered clouds,2026-06-23 12:45:46,Hot
3,London,28.04,68,1019,few clouds,2026-06-23 12:45:46,Hot
4,Rome,21.29,91,1017,overcast clouds,2026-06-23 12:45:46,Moderate


In [47]:
df.to_csv("weather_cleaned.csv", index=False)

In [48]:
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

df.to_sql("weather_data", engine, if_exists="append", index=False)

print("ETL pipeline executed successfully")

ETL pipeline executed successfully
